# MiniGPT Model Training!
Train a miniGPT model on the corbt/all-recipes dataset, to make your own, better than gpt-2 distil (stupid) cookgpt. This one runs better locally! I'm pretty sure you can also run on colab but I haven't tried it yet.

In [ ]:
import gc
import os
import sys
from pathlib import Path

import torch
from datasets import load_dataset

NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.append(str(NOTEBOOK_DIR))

from minigpt import GPTConfig, miniGPT # make sure you import properly

In [ ]:
config = GPTConfig(
    block_size=484,
    n_layers=12,
    n_embd=768,
    n_heads=12,
    dropout=0.1,
)

# No. params: 12 * (768^2 + 768^2 + 768^2) + 768^2 + 768^2 = 84,722,432 parameters

trainer = miniGPT.from_tokenizer_name("gpt2", config=config)

device = "cpu" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
trainer = trainer.to(device)
print(f"Using device: {device}")

In [ ]:
# Load huggingface/all-recipes and split exactly as requested.
raw = load_dataset("corbt/all-recipes")

# Some configs expose only a train split; this handles both cases safely.
if "train" in raw:
    base_split = raw["train"]
else:
    first_key = list(raw.keys())[0]
    base_split = raw[first_key]

split = base_split.train_test_split(test_size=0.05, seed=42)
train_full = split["train"]
test_set = split["test"]

print(f"train_full: {len(train_full):,} samples")
print(f"test_set: {len(test_set):,} samples")

In [ ]:
# Make a smaller training subset so experiments run faster.
subset_size = min(40000, len(train_full))
subset_seed = 42
train_subset = train_full.shuffle(seed=subset_seed).select(range(subset_size))

# Pick a likely text field from the dataset.
candidate_columns = [
    "text",
    "recipe",
    "instructions",
    "directions",
    "title",
    "ingredients",
    "source",
    "input",
]

available_columns = train_subset.column_names
text_column = next((c for c in candidate_columns if c in available_columns), None)
if text_column is None and len(available_columns) == 1:
    text_column = available_columns[0]
if text_column is None:
    raise ValueError(f"No expected text column found. Available: {available_columns}")

def _flatten_to_text(value):
    if value is None:
        return ""
    if isinstance(value, str):
        return value
    if isinstance(value, list):
        return " ".join(_flatten_to_text(v) for v in value)
    if isinstance(value, dict):
        # Prefer common content keys when present.
        for key in ["text", "input", "recipe", "instructions", "directions", "title", "ingredients"]:
            if key in value:
                return _flatten_to_text(value[key])
        return " ".join(f"{k}: {_flatten_to_text(v)}" for k, v in value.items())
    return str(value)

def to_text(row):
    return _flatten_to_text(row.get(text_column)).strip()

train_texts = [to_text(r) for r in train_subset]
train_texts = [t for t in train_texts if t]

print(f"Selected text column: {text_column}")
print(f"train_subset used: {len(train_texts):,} samples")

In [ ]:
# Clear Python and MPS cached objects before training to avoid OOM spikes.
gc.collect()
if device == "mps":
    torch.mps.empty_cache()

# Train on the smaller subset.
trainer.train(
    train_texts,
    epochs=1,
    batch_size=8,
    learning_rate=3e-4,
    warmup_ratio=0.03,
    min_lr_ratio=0.1,
)

output_dir = NOTEBOOK_DIR / "trained_model"
trainer.save(str(output_dir))

print(f"Saved weights to: {output_dir / 'weights.pt'}")
print(f"Saved config to: {output_dir / 'config.json'}")

In [ ]:
output_dir = NOTEBOOK_DIR / "trained_model_small"

print("=" * 60)
print("LOADING MODEL AND GENERATING RECIPES")
print("=" * 60)

try:
    loaded_trainer = miniGPT.load(str(output_dir), map_location=device)
    
    prompts = [
        "Recipe for chocolate ",
        "Ingredients: flour butter sugar",
        "Instructions: preheat oven",
        "Recipe for chocolate",
        "Recipe for chocolate",
        "Recipe for",
        "Gradually stir in",
    ]

    print("=" * 60)
    print("GENERATED RECIPES")
    print("=" * 60)

    for prompt in prompts:
        print(f"\nPrompt: {prompt}")
        print("-" * 60)
        generated = loaded_trainer.generate(prompt, max_length=250, temperature=0.8, top_p=0.95)
        print(generated)
        print()
        
except Exception as e:
    print(f"Error loading model or generating recipes: {e}")